# 缺失值填充方法

本notebook专门介绍几种高级缺失值填充方法：

1. 条件平均值填充

2. 热卡填充（Hot deck imputation）

3. KNN填充

4. 回归填充（Regression）

5. 多重插补（Multiple Imputation）

## 1. 条件平均值填充

与其相似的另一种方法叫**条件平均值填充法**（Conditional Mean Completer）。在该方法中，用于求平均的值并不是从数据集的所有对象中取，而是从与该对象具有**相同决策属性值**的对象中取得。

In [6]:
import pandas as pd
import numpy as np

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)

# 按【性别】分组，填充【年龄】缺失值
df["年龄"] = df.groupby("性别")["年龄"].transform(
    lambda x: x.fillna(x.mean())
)

# 按【性别+城市】双条件分组，填充年龄缺失值
df["年龄"] = df.groupby(["性别", "城市"])["年龄"].transform(
    lambda x: x.fillna(x.mean())
)

# 按【城市】分组，填充【上次消费金额】缺失值
df["上次消费金额"] = df.groupby("城市")["上次消费金额"].transform(
    lambda x: x.fillna(x.mean())
)

print("\n条件平均值填充后的数据：")
print(df)

原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

条件平均值填充后的数据：
   用户ID   姓名 性别         年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.000000    beijing   80000   200.0
1  1002   李四  女  30.000000   Shanghai  120000   500.0
2  1003   王五  男  31.666667  guangzhou   75000   300.0
3  1004   赵六  女  45.000000   shenzhen   95000   280.0
4  1005   孙七  男  30.000000   Shanghai  120000   500.0
5  1006   周八  女  28.000000    Beijing  110000   400.0
6  1007   吴九  男  31.666667   hangzhou   98000   250.0
7  1008  

## 2. 热卡填充（Hot deck imputation，或就近补齐）
-------------------------------------------------------------------

对于一个包含空值的对象，热卡填充法在完整数据中找到**一个**与它最相似的对象，然后用这个**相似对象的值来进行填充**。

不同的问题可能会选用不同的标准来对相似进行判定。该方法概念上很简单，且利用了数据间的关系来进行空值估计。

这个方法的缺点在于难以定义相似标准，**主观因素较多**。

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)


df['城市'] = df['城市'].str.lower()

# 3. 定义热卡填充核心函数
def hot_deck_impute(df, target_col, sim_features):
    """
    热卡填充函数
    :param df: 原始数据集
    :param target_col: 需要填充的目标列
    :param sim_features: 用于计算样本相似度的特征列表（无缺失）
    :return: 填充后的目标列
    """
    # 复制数据，避免修改原数据
    df_impute = df.copy()
    # 筛选：完整样本（目标列无缺失）+ 缺失样本（目标列有缺失）
    complete_samples = df_impute[df_impute[target_col].notna()]
    missing_samples = df_impute[df_impute[target_col].isna()]

    if len(missing_samples) == 0:
        return df_impute[target_col]

    # 对相似度特征做：分类特征独热编码 + 数值特征标准化
    # 提取相似度特征
    sim_data = df_impute[sim_features]
    # 分离分类/数值特征
    cat_cols = sim_data.select_dtypes(include=['object', 'string']).columns.tolist()
    num_cols = sim_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # 独热编码分类特征
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    cat_encoded = encoder.fit_transform(sim_data[cat_cols])
    cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols), index=sim_data.index)

    # 标准化数值特征
    scaler = StandardScaler()
    num_scaled = scaler.fit_transform(sim_data[num_cols])
    num_df = pd.DataFrame(num_scaled, columns=num_cols, index=sim_data.index)

    # 合并最终相似度矩阵
    sim_matrix = pd.concat([cat_df, num_df], axis=1)

    # 遍历所有缺失样本，填充最相似完整样本的值
    for missing_idx in missing_samples.index:
        # 缺失样本的特征向量
        missing_vec = sim_matrix.loc[[missing_idx]]
        # 完整样本的特征向量
        complete_vec = sim_matrix.loc[complete_samples.index]
        # 计算欧氏距离（找最近邻）
        distances = pairwise_distances(missing_vec, complete_vec, metric='euclidean')[0]
        # 最相似的完整样本索引
        most_similar_idx = complete_samples.index[np.argmin(distances)]
        # 用最相似样本的值填充
        df_impute.loc[missing_idx, target_col] = df_impute.loc[most_similar_idx, target_col]

    return df_impute[target_col]

# 4. 配置相似度特征（无缺失的核心特征：性别、城市、年收入）
similarity_features = ['性别', '城市', '年收入']

# 5. 执行热卡填充
df['年龄'] = hot_deck_impute(df, '年龄', similarity_features)
df['上次消费金额'] = hot_deck_impute(df, '上次消费金额', similarity_features)

print("\n热卡填充后的数据：")
print(df)


原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

热卡填充后的数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   shanghai  120000   500.0
2  1003   王五  男  25.0  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000   320.0
4  1005   孙七  男  30.0   shanghai  120000   500.0
5  1006   周八  女  28.0    beijing  110000   400.0
6  1007   吴九  男  35.0   hangzhou   98000   250.0
7  1008   郑十  女  32.0    nanjing  105000   350.0
8  


错误代码及问题
def hot_deck_impute(df, target_col):
    """
    热卡填充法
    参数：
        df: DataFrame
        target_col: 需要填充的列名
    返回：
        填充后的DataFrame
    """
    df_hot = df.copy()

    # 城市编码
    le = LabelEncoder()
    df_hot["城市"] = df_hot["城市"].fillna("未知")
    df_hot["城市"] = le.fit_transform(df_hot["城市"])
    # 性别编码
    df_hot["性别"] = le.fit_transform(df_hot["性别"])

    # 完整数据
    complete = df_hot.dropna()
    # 缺失数据
    missing = df_hot[df_hot[target_col].isnull()]

    if len(missing) == 0:
        return df_hot

    # 计算欧氏距离，找最近1个样本
    dists = pairwise_distances(
        missing.drop(target_col, axis=1),
        complete.drop(target_col, axis=1),
        metric="euclidean"
    )
    # 逐个填充
    for i, idx in enumerate(missing.index):
        nearest = np.argmin(dists[i])
        fill_val = complete.iloc[nearest][target_col]
        df_hot.loc[idx, target_col] = fill_val
    return df_hot

#### 对年龄做热卡填充
df_hot = hot_deck_impute(df, "年龄")
#### 对上次消费金额做热卡填充
df_hot = hot_deck_impute(df_hot, "上次消费金额")

1. 分类特征编码错误（最严重）
代码用 LabelEncoder 编码城市、性别（无序分类特征）
LabelEncoder 会把城市转为 0,1,2,3...，让欧氏距离误以为：
北京(0) < 上海(1) < 广州(2)，给无序特征强加了大小顺序
后果：相似度计算完全失真，找的 “最相似样本” 根本不相似
✅ 标准代码：用 OneHotEncoder 编码无序分类特征，无顺序偏差，计算科学。
2. 未做数值特征标准化
数据中：
年收入：80000~120000（极大数值）
性别 / 城市编码：0/1（极小数值）
欧氏距离会完全被年收入主导，性别、城市对相似度的影响几乎为 0。
✅ 标准代码：对数值特征标准化，消除量纲影响，所有特征公平参与相似度计算。
3. 包含无效干扰特征
代码自动用所有列计算距离，包含：用户ID、姓名（唯一标识，无业务意义）比如用户 ID 1001 和 1002 会因为 ID 数字接近被判定为相似，完全错误。
✅ 标准代码：手动筛选核心相似度特征（性别、城市、年收入），剔除干扰项。
4. 数据未预处理（城市大小写不统一）
原始数据中：beijing / Beijing、Shanghai / shanghai代码会把它们当成两个不同城市，分类错误。
✅ 标准代码：统一城市小写，保证分类准确性。
5. 填充连锁误差
先填充年龄 → 用填充后的年龄再去计算消费金额的相似度填充的估算值会二次干扰结果，误差叠加。
✅ 标准代码：始终用原始完整特征计算相似度，无误差叠加。


## 3. KNN填充
--------------------------------------------------------------------

先根据欧式距离或相关分析来确定距离具有缺失数据样本最近的K个样本，将这K个值加权平均来估计该样本的缺失数据。

根据数据类型的不同，距离度量也不尽相同：
1、连续数据：最常用的距离度量有欧氏距离，曼哈顿距离以及余弦距离。
2、分类数据：汉明（Hamming）距离在这种情况比较常用。对于所有分类属性的取值，如果两个数据点的值不同，则距离加一。汉明距离实际上与属性间不同取值的数量一致。

KNN算法的一个明显缺点是，在分析大型数据集时会变得非常耗时，因为它会在整个数据集中搜索相似数据点。此外，在高维数据集中，最近与最远邻居之间的差别非常小，因此KNN的准确性会降低。

In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# 1. 原始数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})
print("原始数据：")
print(df)
# ===================== 核心修正：预处理 + KNN填充 + 原始格式还原 =====================
# 2. 统一城市大小写（关键：避免Beijing/beijing识别为不同值）
df['城市'] = df['城市'].str.lower()
df_original = df.copy()  # 备份原始结构

# 3. 分离特征：分类特征（性别、城市）+ 数值特征（年龄、年收入、上次消费金额）
cat_cols = ['性别', '城市']       # 分类列
num_cols = ['年龄', '年收入', '上次消费金额']  # 数值列（含缺失值）

# 4. 独热编码分类特征（KNN专用，无顺序偏差）
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df[cat_cols])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols))

# 5. 合并编码后的分类特征 + 数值特征（用于KNN计算）
knn_data = pd.concat([cat_df, df[num_cols]], axis=1)

# 6. KNN填充缺失值（n_neighbors=3，小样本更合适）
imputer = KNNImputer(n_neighbors=3)
knn_filled = imputer.fit_transform(knn_data)
filled_df = pd.DataFrame(knn_filled, columns=knn_data.columns)

# 7. 逆变换 → 把编码还原为原始的【性别、城市】文字
cat_encoded_original = filled_df[encoder.get_feature_names_out(cat_cols)]
cat_original = encoder.inverse_transform(cat_encoded_original)  # 还原分类列
cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)

# 8. 合并所有列，还原成【原始格式的DataFrame】
df_result = df_original.copy()
df_result[cat_cols] = cat_original_df  # 还原性别、城市
df_result[num_cols] = filled_df[num_cols]  # 填充后的数值列

# ===================== 输出最终结果 =====================
print("\nKNN填充后的数据：")
print(df_result.round(0).astype(int, errors='ignore'))  # 年龄/消费金额取整，更美观


原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

KNN填充后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  33  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     283
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  37   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40 

## 4. 回归填充（Regression）
---------------------------------------------------------------------

回归填充法是用已有数据建立回归模型，将缺失值作为因变量进行预测估计。

适用于缺失值与其他变量存在较强线性关系的情况。

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor  # 回归模型（核心）
from sklearn.preprocessing import OneHotEncoder     # 分类特征编码
import warnings
warnings.filterwarnings('ignore')

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)

# ===================== 3. 数据预处理（关键：统一格式+编码） =====================
df_process = df.copy()
# 统一城市大小写（消除beijing/Beijing差异）
df_process['城市'] = df_process['城市'].str.lower()

# 分类特征：性别、城市 → 独热编码（无序特征，无顺序偏差）
cat_features = ['性别', '城市']
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df_process[cat_features])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_features))

# 构建建模用数据集：编码特征 + 数值特征（剔除用户ID、姓名，无业务意义）
num_features = ['年龄', '年收入', '上次消费金额']
model_data = pd.concat([cat_df, df_process[num_features]], axis=1)

# ===================== 4. 定义回归填充通用函数 =====================
def regression_impute(model_data, target_col):
    """
    回归填充函数
    :param model_data: 编码后的建模数据
    :param target_col: 需要填充的目标列
    :return: 填充后的目标列数据
    """
    # 拆分：完整数据（训练集）、缺失数据（预测集）
    train = model_data[model_data[target_col].notna()]
    test = model_data[model_data[target_col].isna()]

    if len(test) == 0:
        return model_data[target_col]

    # 特征X（所有列排除目标列），目标y
    X_train = train.drop(target_col, axis=1)
    y_train = train[target_col]
    X_test = test.drop(target_col, axis=1)

    # 训练随机森林回归模型
    model = RandomForestRegressor(random_state=42, n_estimators=100)
    model.fit(X_train, y_train)

    # 预测缺失值
    model_data.loc[model_data[target_col].isna(), target_col] = model.predict(X_test)
    return model_data[target_col]

# ===================== 5. 执行回归填充 =====================
# 填充 年龄 缺失值
model_data['年龄'] = regression_impute(model_data, '年龄')
# 填充 上次消费金额 缺失值
model_data['上次消费金额'] = regression_impute(model_data, '上次消费金额')

# ===================== 6. 还原原始数据格式（保留性别、城市原始文字） =====================
df_final = df.copy()
df_final['年龄'] = model_data['年龄'].round(0).astype(int)  # 年龄取整
df_final['上次消费金额'] = model_data['上次消费金额'].round(0).astype(int)  # 消费金额取整
df_final['城市'] = df_process['城市']  # 统一大小写后的城市

print("\n回归填充后的数据：")
print(df_final)

原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

回归填充后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  31  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     314
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  31   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40  

## 方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 条件平均值填充 | 考虑了分类变量影响 | 需要有足够的分组样本 | 分组明显的缺失值 |
| 热卡填充 | 保持数据分布 | 相似度定义主观 | 小规模、结构化数据 |
| KNN填充 | 多维考虑，抗噪声 | 计算开销大 | 中等规模数据 |
| 回归填充 | 利用变量关系 | 假设线性关系 | 变量间有较强线性关系 |

## 5. 多重插补（Multiple Imputation）
-------------------------------------------------------------------

多重插补（Multiple Imputation, MI）是一种基于贝叶斯思想的缺失值处理方法。

**核心思想**：对每个缺失值进行**多次（通常3-5次）**插补，产生**多个完整数据集**，然后对每个数据集分别进行分析，最后将结果**汇总**得到最终估计。

**优势**：
- 考虑了缺失值的不确定性
- 能够反映模型参数的变异程度
- 比单次插补（single imputation）更准确、更可靠

**基本步骤**：
1. **插补（Imputation）**：对每个缺失值进行m次插补，生成m个完整数据集
2. **分析（Analysis）**：对每个完整数据集分别进行统计分析（如回归分析）
3. **汇总（Pooling）**：使用Rubin's rules将m个分析结果汇总为最终估计

**常用方法**：
- **MICE（Multivariate Imputation by Chained Equations）**：基于链式方程的多变量插补
- **EM算法 + 马尔可夫链蒙特卡洛（EM+MCMC）**：基于概率模型的插补

In [10]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression, Ridge, BayesianRidge
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)
print(f"\n缺失值统计：\n{df.isnull().sum()}")

原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

缺失值统计：
用户ID      0
姓名        0
性别        0
年龄        2
城市        0
年收入       0
上次消费金额    1
dtype: int64


### 5.1 MICE（链式方程多变量插补）

MICE通过建立**条件模型**来迭代地填充缺失值。

**算法流程**：
1. 用均值初始化所有缺失值
2. 选择一个有缺失值的变量作为目标
3. 用其他变量预测目标变量的缺失值（建立回归模型）
4. 用预测值填补目标变量的缺失值
5. 重复步骤2-4，遍历所有有缺失值的变量
6. 重复整个过程多次（通常10-20次迭代）

sklearn的IterativeImputer实现了类似MICE的算法，默认使用BayesianRidge回归。

In [11]:
def mice_impute(df, n_imputations=5, n_iterations=10, random_state=42):
    """
    MICE多重插补
    参数：
        df: 包含缺失值的DataFrame
        n_imputations: 插补次数（生成n_imputations个完整数据集）
        n_iterations: 每次插补的迭代次数
        random_state: 随机种子
    返回：
        多个插补后的DataFrame列表
    """
    np.random.seed(random_state)
    
    df_numeric = df.select_dtypes(include=[np.number]).copy()
    
    imputed_datasets = []
    
    for i in range(n_imputations):
        df_temp = df_numeric.copy()
        
        for iteration in range(n_iterations):
            for col in df_temp.columns:
                if df_temp[col].isnull().sum() > 0:
                    other_cols = [c for c in df_temp.columns if c != col]
                    
                    known_mask = df_temp[col].notna()
                    unknown_mask = df_temp[col].isna()
                    
                    if known_mask.sum() > 0 and unknown_mask.sum() > 0:
                        X_train = df_temp.loc[known_mask, other_cols]
                        y_train = df_temp.loc[known_mask, col]
                        X_pred = df_temp.loc[unknown_mask, other_cols]
                        
                        model = Ridge(alpha=1.0)
                        model.fit(X_train, y_train)
                        
                        predictions = model.predict(X_pred)
                        noise = np.random.normal(0, y_train.std() * 0.1, len(predictions))
                        df_temp.loc[unknown_mask, col] = predictions + noise
        
        imputed_datasets.append(df_temp.copy())
        print(f"第 {i+1} 次插补完成")
    
    return imputed_datasets


print("执行MICE多重插补（5次插补）...\n")
imputed_datasets = mice_impute(df, n_imputations=5, n_iterations=10, random_state=42)

print("\n" + "="*60)
print("各次插补结果对比：")
print("="*60)
for i, imp_df in enumerate(imputed_datasets):
    print(f"\n【第 {i+1} 次插补】年龄列: {imp_df['年龄'].values}")
    print(f"【第 {i+1} 次插补】上次消费金额列: {imp_df['上次消费金额'].values}")

执行MICE多重插补（5次插补）...



ValueError: Input X contains NaN.
Ridge does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### 5.2 使用sklearn的IterativeImputer（贝叶斯岭回归）

sklearn提供的IterativeImputer是一种**单变量迭代插补**方法，对每个有缺失值的变量依次进行插补，使用其他变量作为特征建立回归模型。

In [ ]:
df_sklearn = df.select_dtypes(include=[np.number]).copy()

print("用于插补的数值型数据：")
print(df_sklearn)

imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    random_state=42,
    verbose=0
)

df_imputed_sklearn = pd.DataFrame(
    imputer.fit_transform(df_sklearn),
    columns=df_sklearn.columns
)

print("\nIterativeImputer填充后的数据：")
print(df_imputed_sklearn)

### 5.3 使用RandomForest进行多重插补

使用随机森林作为迭代插补的基模型，能够捕捉非线性关系。

In [ ]:
def multiple_imputation_rf(df, n_imputations=5, random_state=42):
    """
    基于随机森林的多重插补
    参数：
        df: DataFrame
        n_imputations: 插补次数
        random_state: 随机种子
    返回：
        imputed_dfs: 多个插补后的DataFrame列表
        pooled_results: 汇总统计结果
    """
    np.random.seed(random_state)
    
    df_numeric = df.select_dtypes(include=[np.number]).copy()
    imputed_datasets = []
    
    for i in range(n_imputations):
        df_temp = df_numeric.copy()
        
        for col in df_temp.columns:
            if df_temp[col].isnull().sum() > 0:
                other_cols = [c for c in df_temp.columns if c != col]
                
                known_mask = df_temp[col].notna()
                unknown_mask = df_temp[col].isna()
                
                if known_mask.sum() > 1 and unknown_mask.sum() > 0:
                    X_train = df_temp.loc[known_mask, other_cols]
                    y_train = df_temp.loc[known_mask, col]
                    X_pred = df_temp.loc[unknown_mask, other_cols]
                    
                    rf_model = RandomForestRegressor(n_estimators=10, random_state=random_state+i)
                    rf_model.fit(X_train, y_train)
                    
                    predictions = rf_model.predict(X_pred)
                    df_temp.loc[unknown_mask, col] = predictions
        
        imputed_datasets.append(df_temp.copy())
    
    pooled_df = pd.concat(imputed_datasets, axis=0).groupby(level=0).mean()
    
    return imputed_datasets, pooled_df


print("执行基于随机森林的多重插补（5次）...\n")
rf_imputed_datasets, rf_pooled = multiple_imputation_rf(df, n_imputations=5, random_state=42)

print("【随机森林多重插补】各次插补结果对比：")
for i, imp_df in enumerate(rf_imputed_datasets):
    print(f"第{i+1}次 - 年龄: {imp_df['年龄'].values}")

print("\n【汇总结果（均值）】：")
print(rf_pooled[['年龄', '上次消费金额']])

### 5.4 汇总分析（Pooling）

根据Rubin's rules，多重插补的汇总规则：

**点估计**：各次插补结果的**均值**

**方差估计**：
- 组内方差（Within-imputation variance）：各次插补结果方差的均值
- 组间方差（Between-imputation variance）：各次插补结果之间方差
- 总方差 = 组内方差 + 组间方差 + 组间方差/m（m为插补次数）

**自由度**：使用Rubin's rules公式计算

In [ ]:
def pool_multiple_imputations(imputed_datasets, column_name):
    """
    汇总多重插补结果
    参数：
        imputed_datasets: 多个DataFrame组成的列表
        column_name: 要汇总的列名
    返回：
        dict: 包含点估计和标准误的字典
    """
    m = len(imputed_datasets)
    
    imp_values = [df[column_name].values for df in imputed_datasets]
    
    Q_bar = np.mean([v.mean() for v in imp_values])
    
    within_var = np.mean([v.var() for v in imp_values])
    between_var = np.var([v.mean() for v in imp_values])
    
    total_var = within_var + (1 + 1/m) * between_var
    total_std = np.sqrt(total_var)
    
    return {
        'point_estimate': Q_bar,
        'within_variance': within_var,
        'between_variance': between_var,
        'total_variance': total_var,
        'standard_error': total_std,
        'confidence_interval_95': (Q_bar - 1.96 * total_std, Q_bar + 1.96 * total_std)
    }


print("="*60)
print("汇总分析结果（Rubin's rules）")
print("="*60)

for col in ['年龄', '上次消费金额']:
    result = pool_multiple_imputations(rf_imputed_datasets, col)
    print(f"\n【{col}】")
    print(f"  点估计（均值）: {result['point_estimate']:.2f}")
    print(f"  组内方差: {result['within_variance']:.4f}")
    print(f"  组间方差: {result['between_variance']:.4f}")
    print(f"  总方差: {result['total_variance']:.4f}")
    print(f"  标准误: {result['standard_error']:.2f}")
    print(f"  95%置信区间: ({result['confidence_interval_95'][0]:.2f}, {result['confidence_interval_95'][1]:.2f})")

### 5.5 不同方法对比

In [ ]:
print("="*70)
print("各插补方法结果对比（年龄列）")
print("="*70)

df_compare = pd.DataFrame({
    '用户ID': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    '原始年龄': [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    'MICE_插补1': rf_imputed_datasets[0]['年龄'].values,
    'MICE_插补2': rf_imputed_datasets[1]['年龄'].values,
    'MICE_插补3': rf_imputed_datasets[2]['年龄'].values,
    'MICE_汇总': rf_pooled['年龄'].values,
    'IterativeImputer': df_imputed_sklearn['年龄'].values
})

print(df_compare.to_string(index=False))

print("\n" + "="*70)
print("各方法统计量对比")
print("="*70)

stats_comparison = pd.DataFrame({
    '方法': ['原始数据(排除NaN)', 'MICE汇总', 'IterativeImputer'],
    '均值': [
        df['年龄'].mean(),
        rf_pooled['年龄'].mean(),
        df_imputed_sklearn['年龄'].mean()
    ],
    '标准差': [
        df['年龄'].std(),
        rf_pooled['年龄'].std(),
        df_imputed_sklearn['年龄'].std()
    ],
    '中位数': [
        df['年龄'].median(),
        rf_pooled['年龄'].median(),
        df_imputed_sklearn['年龄'].median()
    ]
})

print(stats_comparison.to_string(index=False))